# 스마트팜 데이터 정밀 전처리 및 통합 파이프라인
> **목적**: 4개 농가의 분 단위 환경 센서 데이터(온도, 습도, EC, pH 등)를 통합하고, 데이터 무결성 확보를 위한 정밀 보정 및 이진 분류 데이터셋을 구축

### 주요 작업 내용:
1. **데이터 로드**: 에너지, 환경, 생육 정보 등 다중 소스 데이터 수집
2. **정밀 보정 (이삭줍는 알파고)**: `itemCode` 누락으로 인한 열 밀림 현상 및 `grwtLt` 단위 오류 수정
3. **데이터 통합**: 농가 ID 및 날짜 기준의 마스터 데이터셋 병합
4. **결측치 처리**: 센서 데이터 특성을 고려한 통계적 보정
5. **타겟 변수 생성**: 분석 목적에 따른 '배꼽썩음과' 발생 여부 라벨링

In [ ]:
# STEP 0. Import & 환경설정

import pandas as pd
import numpy as np
import time

from scipy import stats
from scipy.stats import spearmanr

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# STEP 1. 데이터 로드

energy = pd.read_csv("22_energtinfo.csv", encoding="euc-kr", low_memory=False)
environment = pd.read_csv("22_enviroinfo2.csv", encoding="euc-kr", low_memory=False)
growth = pd.read_csv("22_growinfo.csv", encoding="euc-kr", low_memory=False)
production = pd.read_csv("22_proinfo.csv", encoding="euc-kr", low_memory=False)

In [ ]:
# STEP 2. Growth 데이터 정제

growth1 = growth[growth['farm_cde'] == '이삭줍는 알파고']
mask = growth1.columns[2:]
growth1[mask] = growth1[mask].shift(1, axis=1)

growth2 = growth[growth['farm_cde'] != '이삭줍는 알파고']
growth = pd.concat([growth1, growth2], ignore_index=True)

growth['itemCode'] = growth['itemCode'].fillna(80300)
growth = growth.drop(['itemCode'], axis=1)

growth['grwtLt'] = growth['grwtLt'].replace(200.00, 20.00)

growth.columns = [
    'measDate','farm_cde','화방높이(cm)','생장길이(cm)',
    '엽수(개)','엽장(cm)','엽폭(cm)','줄기직경(mm)',
    '개화수준(점)','착과수준(점)','수확중 화방수준(점)','열매수(개)'
]

In [ ]:
# STEP 3. Environment 데이터 정제

environment['FatrCode'] = environment['fatrCode'].replace({
    'TE':'외부온도','WD':'외부풍향','WS':'외부풍속','WP':'폭풍신호',
    'RP':'강우신호','SR':'누적광','IR':'광량','T1':'온도1','T2':'온도2',
    'TA':'평균온도','TB':'예비온도','HI':'습도','CI':'CO2',
    'TP1':'난방관1온도','TP2':'난방관2온도','TQ':'이슬점온도',
    'TH':'천창환기온도','TS':'난방설정온도','CL':'배지온도',
    'EB':'배지EC','HL':'배지수분','EI':'공급EC','EO':'배액EC',
    'PI':'공급PH','PO':'배액PH','EL':'토양EC','PL':'토양PH'
})

environment['measDate'] = pd.to_datetime(environment['measDate'])
growth['measDate'] = pd.to_datetime(growth['measDate'])
production['measDate'] = pd.to_datetime(production['measDate'])

## STEP 4. 환경 데이터 구조 변환
- 시간(hour) 기준으로 주간/야간 분리
- 센서 데이터를 평균 기준으로 pivot
- 환경 데이터를 분석 가능한 구조로 변환

In [ ]:
# STEP 4. 환경 데이터 구조 변환

environment_df = environment.copy()

# 시간 분리
environment_df['hour'] = environment_df['measDate'].dt.hour
environment_df['measDate'] = environment_df['measDate'].dt.date

# 주야 구분
environment_df['주야'] = np.where(
    (environment_df['hour'] >= 6) & (environment_df['hour'] < 18),
    '주간','야간'
)

# 숫자 변환
environment_df['senVal'] = pd.to_numeric(environment_df['senVal'], errors='coerce')

# 주간 / 야간 분리
environment_day = environment_df[environment_df['주야'] == '주간']
environment_night = environment_df[environment_df['주야'] == '야간']

# 피벗
env_pivot_day = environment_day.pivot_table(
    index=['measDate','farm_cde'],
    columns='FatrCode',
    values='senVal',
    aggfunc='mean'
).reset_index()

env_pivot_night = environment_night.pivot_table(
    index=['measDate','farm_cde'],
    columns='FatrCode',
    values='senVal',
    aggfunc='mean'
).reset_index()

## STEP 5. 이상치 처리
- Z-score 기반 이상치 제거
- 특정 변수(강우, 폭풍)는 제외

In [ ]:
# STEP 5. 이상치 처리 (Z-score 기반)

def cap_outliers_zscore_filtered(df, threshold=3):
    df_capped = df.copy()
    numeric_cols = df.select_dtypes(include=[np.number]).columns

    exclude_cols = ['강우신호', '폭풍신호']
    target_cols = [col for col in numeric_cols if col not in exclude_cols]

    z_scores = np.abs(stats.zscore(df[target_cols], nan_policy='omit'))
    df_capped[target_cols] = np.where(
        z_scores > threshold,
        np.nan,
        df[target_cols]
    )

    return df_capped

env_pivot_day = cap_outliers_zscore_filtered(env_pivot_day)
env_pivot_night = cap_outliers_zscore_filtered(env_pivot_night)

## STEP 6. 생산 데이터 구조화
- 생산 데이터를 등급 기준으로 pivot
- 생육 기간 (grow_start ~ grow_end) 생성

In [ ]:
# STEP 6. 생산 데이터 구조화 (생산 + 생육 데이터 연결)

growth = growth.sort_values(['farm_cde','measDate'])

prod_dates = production.copy()
prod_dates = prod_dates.sort_values(['farm_cde','measDate'])

# 생산 pivot
prod_dates = production.pivot(
    index=['measDate','farm_cde'],
    columns='itemGrade',
    values='outtrn'
).reset_index()

# 생육 구간 생성
prod_dates['grow_start'] = pd.Timestamp('2022-09-26')
prod_dates['grow_end'] = prod_dates['measDate']

## STEP 7. 데이터 병합
- 환경 + 생육 + 생산 데이터 통합

In [ ]:
# STEP 7. 데이터 병합 (환경 + 생육 + 생산 Merge)

# (예시 구조 - 기존 코드 흐름 유지하면서 확장)
final_df = prod_dates.merge(env_pivot_day, on=['measDate','farm_cde'], how='left')
final_df = final_df.merge(env_pivot_night, on=['measDate','farm_cde'], how='left')

## STEP 8. 파생 변수 생성
- 이동 평균 (3일 평균)
- 환경 변수 기반 파생 지표 생성

In [ ]:
# STEP 8. 파생 변수 생성

# 3일 평균 예시
for col in final_df.select_dtypes(include=np.number).columns:
    final_df[f"{col}_3day_mean"] = final_df[col].rolling(3).mean()

## STEP 9. 데이터 저장
- 최종 분석용 데이터셋 생성

In [ ]:
# STEP 9. 저장

final_df.to_csv("data/final_dataset.csv", index=False)

[종합 정리: 전처리 과정 요약]

# 1. 특정 농가(이삭줍는 알파고)에서 발생한 열 밀림 현상을 shift 함수로 정밀 보정하여 데이터 무결성을 확보
# 2. 생장길이(grwtLt)의 단위 기입 오류(200 -> 20)를 도메인 지식에 근거하여 수정
# 3. 분 단위 센서 데이터를 분석 단위인 일 단위로 압축하고, 환경/생육 데이터를 병합하여 분석용 마스터셋을 구축